In [ ]:
import requests
from datetime import datetime, timedelta


today = datetime.today()

from_date = (today - timedelta(days=15)).strftime('%Y-%m-%d')
to_date = (today + timedelta(days=15)).strftime('%Y-%m-%d')

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'ko-KR,ko;q=0.9,zh-CN;q=0.8,zh;q=0.7,en-US;q=0.6,en;q=0.5,ja;q=0.4',
    'charset': 'utf-8',
    'origin': 'https://m.sports.naver.com',
    'priority': 'u=1, i',
    'referer': 'https://m.sports.naver.com/kbaseball/schedule/index?category=kbo&date=2026-03-12',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-sports-backend': 'kotlin',
}

league = 'kbo'

league_info = {
    'kbo': {'upperCategoryId': 'kbaseball', 'categoryId': 'kbo'},
    'npb': {'upperCategoryId': 'wbaseball', 'categoryId': 'npb'},
    'mlb': {'upperCategoryId': 'wbaseball', 'categoryId': 'mlb'}
}

params = {
    'fields': 'basic,schedule,baseball,manualRelayUrl',
    'upperCategoryId': league_info[league]['upperCategoryId'],
    'categoryId': league_info[league]['categoryId'],
    'fromDate': from_date,
    'toDate': to_date,
    'roundCodes': '',
    'size': '500',
}


def safe_int(value, default=0):
    try:
        return int(value)
    except:
        return default


def get_search_end_inning(inn_value):
    """
    예:
    '3'   -> 3
    '3 ⅓' -> 4
    '3 ⅔' -> 4
    '4'   -> 4
    '4 ⅔' -> 5
    """
    if inn_value in [None, '']:
        return 0

    s = str(inn_value).strip()
    s = ' '.join(s.split())

    base_str = s.split()[0]

    try:
        base_inning = int(base_str)
    except:
        return 0

    if ('⅓' in s) or ('⅔' in s) or ('1/3' in s) or ('2/3' in s):
        return base_inning + 1

    return base_inning


def get_result_type(text):
    text = str(text).strip()

    if ('몸에 맞는 볼' in text) or ('사구' in text) or ('몸맞는볼' in text):
        return 'hbp'

    if ('볼넷' in text) or ('4구' in text) or ('고의4구' in text):
        return 'bb'

    return ''


def is_runner_event_text(text):
    text = str(text).strip()
    runner_keywords = ['루주자', '진루', '도루', '폭투', '보크', '견제']
    return any(keyword in text for keyword in runner_keywords)


def make_batter_code_name_map(record_data):
    batter_map = {}

    for side in ['away', 'home']:
        batters = record_data.get('battersBoxscore', {}).get(side, [])
        for batter in batters:
            code = str(
                batter.get('playerCode', batter.get('pcode', batter.get('batterCode', '')))
            ).strip()
            name = batter.get('name', '')
            if code:
                batter_map[code] = name

    return batter_map


def find_final_batter_result(text_options):
    """
    뒤에서부터 보면서
    주자 진루 같은 부가 이벤트를 제외하고
    볼넷/사구 최종 결과를 찾는다.
    """
    if not isinstance(text_options, list) or not text_options:
        return None

    for option in reversed(text_options):
        text = str(option.get('text', '')).strip()
        current_game_state = option.get('currentGameState', {})

        pitcher = str(current_game_state.get('pitcher', '')).strip()
        batter = str(current_game_state.get('batter', '')).strip()

        if not text:
            continue
        if is_runner_event_text(text):
            continue
        if not pitcher or not batter:
            continue

        result_type = get_result_type(text)
        if result_type in ['bb', 'hbp']:
            return {
                'result_type': result_type,
                'text': text,
                'pitcher': pitcher,
                'batter': batter
            }

    return None


def extract_walks_from_one_inning(
    relay_data,
    inning,
    away_starter_pcode,
    home_starter_pcode,
    batter_code_name_map
):
    away_walks = []
    home_walks = []

    text_relays = relay_data.get('result', {}).get('textRelayData', {}).get('textRelays', [])

    for relay in text_relays:
        title = str(relay.get('title', '')).strip()
        text_options = relay.get('textOptions', [])

        final_result = find_final_batter_result(text_options)
        if not final_result:
            continue

        pitcher_code = str(final_result.get('pitcher', '')).strip()
        batter_code = str(final_result.get('batter', '')).strip()
        text = str(final_result.get('text', '')).strip()
        result_type = final_result.get('result_type', '')

        batter_name = batter_code_name_map.get(batter_code, '')

        row = {
            'inning': inning,
            'pitcher_code': pitcher_code,
            'batter_code': batter_code,
            'batter_name': batter_name,
            'title': title,
            'text': text,
            'result_type': result_type
        }

        if pitcher_code == str(away_starter_pcode):
            away_walks.append(row)

        elif pitcher_code == str(home_starter_pcode):
            home_walks.append(row)

    return away_walks, home_walks


def is_inning_start_title(title):
    """
    예: '2회초 한화 공격', '3회말 LG 공격'
    """
    title = str(title).strip()
    return ('회초' in title or '회말' in title) and ('공격' in title)


def get_first_pitch_result_of_first_batter(relay_data, inning, batter_code_name_map):
    """
    각 이닝 relay에서
    '공격 시작 안내' 바로 다음 타자의 첫 공 결과(pitchNum == 1) 추출
    """
    text_relays = relay_data.get('result', {}).get('textRelayData', {}).get('textRelays', [])

    if not text_relays or len(text_relays) < 2:
        return None

    start_idx = None

    # 네가 확인한 구조상 끝쪽에 '2회초 한화 공격' 같은 안내가 있음
    # 고정 -1/-2로 가지 않고 실제 제목으로 찾는다
    for idx in range(len(text_relays) - 1, -1, -1):
        title = str(text_relays[idx].get('title', '')).strip()
        if is_inning_start_title(title):
            start_idx = idx
            break

    if start_idx is None:
        return None

    # 공격 시작 안내 다음 타자
    batter_idx = start_idx - 1
    if batter_idx < 0:
        return None

    first_batter_relay = text_relays[batter_idx]
    title = str(first_batter_relay.get('title', '')).strip()
    text_options = first_batter_relay.get('textOptions', [])

    if not isinstance(text_options, list) or not text_options:
        return None

    for option in text_options:
        if option.get('pitchNum') == 1:
            current_game_state = option.get('currentGameState', {})
            batter_code = str(current_game_state.get('batter', '')).strip()
            batter_name = batter_code_name_map.get(batter_code, '')

            return {
                'inning': inning,
                'title': title,
                'batter_code': batter_code,
                'batter_name': batter_name,
                'pitchResult': option.get('pitchResult', ''),
                'text': option.get('text', ''),
                'stuff': option.get('stuff', ''),
                'speed': option.get('speed', '')
            }

    return None


response = requests.get(
    'https://api-gw.sports.naver.com/schedule/games',
    params=params,
    headers=headers
)

games = response.json()['result']['games']

all_rows = []
first_pitch_rows = []

for game in games:
    game_id = game.get('gameId', '')
    game_date = game.get('gameDate', '')
    home_team = game.get('homeTeamName', '')
    away_team = game.get('awayTeamName', '')
    home_starting = game.get('homeStarterName', '')
    away_starting = game.get('awayStarterName', '')
    status_code = game.get('statusCode', '')

    if status_code != 'RESULT':
        continue

    record_url = f'https://api-gw.sports.naver.com/schedule/games/{game_id}/record'
    record_response = requests.get(record_url, headers=headers)
    record_json = record_response.json()

    if 'result' not in record_json or 'recordData' not in record_json['result']:
        continue

    record_data = record_json['result']['recordData']

    away_pitchers = record_data.get('pitchersBoxscore', {}).get('away', [])
    home_pitchers = record_data.get('pitchersBoxscore', {}).get('home', [])

    if not away_pitchers or not home_pitchers:
        continue

    away_pitcher = away_pitchers[0]
    home_pitcher = home_pitchers[0]

    kor_away_pitcher_record = {
        'name': away_pitcher.get('name', away_starting),
        'inn': away_pitcher.get('inn', ''),
        'hit': away_pitcher.get('hit', ''),
        'r': away_pitcher.get('r', ''),
        'kk': away_pitcher.get('kk', ''),
        'bb': away_pitcher.get('bb', ''),
        'ab': away_pitcher.get('ab', ''),
        'bf': away_pitcher.get('bf', ''),
        'era': away_pitcher.get('era', ''),
        'w_l': away_pitcher.get('wls', ''),
        'pcode': str(away_pitcher.get('pcode', away_pitcher.get('playerCode', ''))).strip()
    }

    kor_home_pitcher_record = {
        'name': home_pitcher.get('name', home_starting),
        'inn': home_pitcher.get('inn', ''),
        'hit': home_pitcher.get('hit', ''),
        'r': home_pitcher.get('r', ''),
        'kk': home_pitcher.get('kk', ''),
        'bb': home_pitcher.get('bb', ''),
        'ab': home_pitcher.get('ab', ''),
        'bf': home_pitcher.get('bf', ''),
        'era': home_pitcher.get('era', ''),
        'w_l': home_pitcher.get('wls', ''),
        'pcode': str(home_pitcher.get('pcode', home_pitcher.get('playerCode', ''))).strip()
    }

    away_starter_pcode = kor_away_pitcher_record['pcode']
    home_starter_pcode = kor_home_pitcher_record['pcode']

    if not away_starter_pcode or not home_starter_pcode:
        continue

    away_search_end_inning = get_search_end_inning(kor_away_pitcher_record['inn'])
    home_search_end_inning = get_search_end_inning(kor_home_pitcher_record['inn'])
    max_search_inning = max(away_search_end_inning, home_search_end_inning)

    if max_search_inning == 0:
        continue

    batter_code_name_map = make_batter_code_name_map(record_data)

    away_walks_all = []
    home_walks_all = []

    print('=' * 100)
    print(f'{game_date}  {away_team} vs {home_team}')
    print()

    for inning in range(1, max_search_inning + 1):
        relay_url = f'https://api-gw.sports.naver.com/schedule/games/{game_id}/relay?inning={inning}'
        relay_response = requests.get(relay_url, headers=headers)

        try:
            relay_data = relay_response.json()
        except:
            continue

        # 기존 볼넷/사구 추출
        away_walks, home_walks = extract_walks_from_one_inning(
            relay_data=relay_data,
            inning=inning,
            away_starter_pcode=away_starter_pcode,
            home_starter_pcode=home_starter_pcode,
            batter_code_name_map=batter_code_name_map
        )

        away_walks_all.extend(away_walks)
        home_walks_all.extend(home_walks)

        # 추가: 각 이닝 첫 타자 첫 공 결과 추출
        first_pitch_info = get_first_pitch_result_of_first_batter(
            relay_data=relay_data,
            inning=inning,
            batter_code_name_map=batter_code_name_map
        )

        if first_pitch_info:
            print(
                f'{inning}회 첫타자 첫공 -> '
                f'{first_pitch_info["batter_name"]} / '
                f'{first_pitch_info["pitchResult"]} / '
                f'{first_pitch_info["text"]}'
            )

            first_pitch_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'inning': inning,
                'away_team': away_team,
                'home_team': home_team,
                'batter_name': first_pitch_info['batter_name'],
                'batter_code': first_pitch_info['batter_code'],
                'title': first_pitch_info['title'],
                'pitchResult': first_pitch_info['pitchResult'],
                'text': first_pitch_info['text'],
                'stuff': first_pitch_info['stuff'],
                'speed': first_pitch_info['speed']
            })

    # 공식 BB(볼넷+사구 포함 가능) 수만큼 잘라주기
    away_official_bb = safe_int(kor_away_pitcher_record['bb'], 0)
    home_official_bb = safe_int(kor_home_pitcher_record['bb'], 0)

    away_walks_all = away_walks_all[:away_official_bb]
    home_walks_all = home_walks_all[:home_official_bb]

    print()
    print(f'[원정 선발] {kor_away_pitcher_record["name"]}')
    print(f'이닝: {kor_away_pitcher_record["inn"]} / 공식 BB: {kor_away_pitcher_record["bb"]}')
    if away_walks_all:
        for row in away_walks_all:
            print(f'  {row["inning"]}회 - {row["batter_name"]} - {row["result_type"]} - {row["text"]}')
            all_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'team_type': 'away',
                'pitcher_name': kor_away_pitcher_record['name'],
                'pitcher_pcode': away_starter_pcode,
                'pitcher_inn': kor_away_pitcher_record['inn'],
                'pitcher_bb': kor_away_pitcher_record['bb'],
                'walk_inning': row['inning'],
                'batter_name': row['batter_name'],
                'batter_code': row['batter_code'],
                'result_type': row['result_type'],
                'text': row['text'],
                'away_team': away_team,
                'home_team': home_team
            })
    else:
        print('  없음')

    print(f'추출 BB/HBP: {len(away_walks_all)}')
    print()

    print(f'[홈 선발] {kor_home_pitcher_record["name"]}')
    print(f'이닝: {kor_home_pitcher_record["inn"]} / 공식 BB: {kor_home_pitcher_record["bb"]}')
    if home_walks_all:
        for row in home_walks_all:
            print(f'  {row["inning"]}회 - {row["batter_name"]} - {row["result_type"]} - {row["text"]}')
            all_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'team_type': 'home',
                'pitcher_name': kor_home_pitcher_record['name'],
                'pitcher_pcode': home_starter_pcode,
                'pitcher_inn': kor_home_pitcher_record['inn'],
                'pitcher_bb': kor_home_pitcher_record['bb'],
                'walk_inning': row['inning'],
                'batter_name': row['batter_name'],
                'batter_code': row['batter_code'],
                'result_type': row['result_type'],
                'text': row['text'],
                'away_team': away_team,
                'home_team': home_team
            })
    else:
        print('  없음')

    print(f'추출 BB/HBP: {len(home_walks_all)}')
    print()